# 02 · Formatting & styles

A tour of the built-in **formats** (per-row cell rendering) and
**styles/themes** (whole-table look). The two are independent: the
stored value is always the raw number.

## A model that exercises many formats

Each row uses a different `PredefinedFormats` value so we can see how
they render side by side.

In [ ]:
from dataclasses import dataclass

from finmodel import Model, row, PredefinedFormats as F, PredefinedStyles as S



@dataclass

class Inputs:

    revenue: float

    growth: float



class Showcase(Model[Inputs]):

    @row(group="Scaled", format=F.MILLIONS)

    def revenue_m(self, t):

        return (self.inputs.revenue if t == 0 else self.revenue_m(t-1) * (1 + self.inputs.growth))



    @row(group="Currency", format=F.USD)

    def revenue_usd(self, t):

        return self.revenue_m(t)



    @row(group="Currency", format=F.EUR)

    def revenue_eur(self, t):

        return self.revenue_m(t) * 0.92



    @row(group="Ratios", format=F.PERCENTAGE)

    def growth(self, t):

        return self.inputs.growth



    @row(group="Ratios", format=F.PERCENTAGE_SIGNED)

    def yoy_change(self, t):

        return 0.0 if t == 0 else self.revenue_m(t)/self.revenue_m(t-1) - 1



    @row(group="Ratios", format=F.BASIS_POINTS)

    def spread(self, t):

        return 0.0125



    @row(group="Ratios", format=F.MULTIPLE)

    def ev_ebitda(self, t):

        return 8.5



    @row(group="Flags", format=F.BOOLEAN)

    def above_target(self, t):

        return self.revenue_m(t) > 1.3



    @row(group="Totals", format=F.TOTAL)

    def revenue_total(self, t):

        return self.revenue_m(t)

In [ ]:
inputs = Inputs(revenue=1.0, growth=0.10)

model = Showcase(periods=5, inputs=inputs)

model.calculate()

model.show()

## The same model in every theme

Pass any `PredefinedStyles` value as `style=`. Below we re-render the
model in each built-in theme.

In [ ]:
from IPython.display import display, Markdown



themes = [

    "CLASSIC_LIGHT", "MINIMAL", "CORPORATE_BLUE", "EMERALD_LIGHT",

    "JETBRAINS_LIGHT_THEME", "JETBRAINS_DARK_THEME", "SLATE_DARK",

    "TERMINAL", "PRINT",

]

for name in themes:

    m = Showcase(periods=5, inputs=inputs, style=getattr(S, name))

    m.calculate()

    display(Markdown(f"### {name}"))

    display(m.show())

## Custom formats and themes

A format is any `value -> str` callable wrapped in `Format`. A theme is
a `Style` with your own colours. See `docs/formatting.md` and
`docs/styling.md` for the full field reference.

In [ ]:
from finmodel import Format, Style

from finmodel.styles import make_currency_formatter



# Custom format: Swiss francs with a prefixed symbol.

CHF = Format(make_currency_formatter("CHF ", decimals=0))



# Custom theme: warm amber-on-charcoal.

AMBER_DARK = Style(

    default="background-color: #1c1917; color: #fbbf24;",

    alternate="background-color: #232020; color: #fbbf24;",

    highlight="background-color: #44403c; color: #fde68a; font-weight: bold;",

    bool_true="background-color: #14532d; color: #86efac; font-weight: bold;",

    bool_false="background-color: #7f1d1d; color: #fca5a5; font-weight: bold;",

    column_header="background-color: #0c0a09; color: #fde68a; font-weight: bold;",

    index_header_l0="background-color: #0c0a09; color: #fde68a; font-weight: bold;",

    blank_header="background-color: #0c0a09;",

    global_cells=[("font-family", "monospace"), ("font-size", "12px"),

                  ("padding", "6px 14px"), ("white-space", "nowrap")],

    show_category_title=False, show_concept_title=False,

)



class Mini(Model[Inputs]):

    @row(format=CHF)

    def revenue(self, t):

        return 1000 * (1 + self.inputs.growth) ** t



m = Mini(periods=4, inputs=inputs, style=AMBER_DARK)

m.calculate()

m.show()